In [3]:
# migrate_hnsw_indexes.py
import os
import sys
import psycopg2

SQLS = [
    "DROP INDEX IF EXISTS idx_article_section_embedding_embedding;",
    """
    CREATE INDEX IF NOT EXISTS idx_article_section_embedding_hnsw
    ON article_section_embedding
    USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64);
    """,
    "DROP INDEX IF EXISTS idx_protocol_embedding_embedding;",
    """
    CREATE INDEX IF NOT EXISTS idx_protocol_embedding_hnsw
    ON protocol_embedding
    USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64);
    """,
]

def main():
    # 환경변수 우선 (없으면 각자 맞게 DSN 직접 넣어도 됨)
    dsn = os.getenv("POSTGRES_DSN")
    if not dsn:
        host = os.getenv("POSTGRES_HOST", "localhost")
        port = os.getenv("POSTGRES_PORT", "5432")
        db   = os.getenv("POSTGRES_DB", "sknfinaldb")
        user = os.getenv("POSTGRES_USER", "postgres")
        pw   = os.getenv("POSTGRES_PASSWORD", "postgres")
        dsn = f"host={host} port={port} dbname={db} user={user} password={pw}"

    conn = psycopg2.connect(dsn)
    conn.autocommit = False

    try:
        with conn.cursor() as cur:
            # (선택) pgvector 버전 확인 (HNSW는 pgvector v0.5.0+에서 지원)
            cur.execute("SELECT extversion FROM pg_extension WHERE extname='vector';")
            row = cur.fetchone()
            if not row:
                raise RuntimeError("pgvector extension(vector)이 설치되어 있지 않습니다.")
            print(f"[OK] pgvector version: {row[0]}")

            for sql in SQLS:
                cur.execute(sql)
                print("[OK] executed:", " ".join(sql.split())[:120], "...")
        conn.commit()
        print("\n✅ Done: HNSW index migration completed.")
    except Exception as e:
        conn.rollback()
        print("\n❌ Failed. Rolled back.")
        raise
    finally:
        conn.close()

if __name__ == "__main__":
    main()


[OK] pgvector version: 0.8.1
[OK] executed: DROP INDEX IF EXISTS idx_article_section_embedding_embedding; ...
[OK] executed: CREATE INDEX IF NOT EXISTS idx_article_section_embedding_hnsw ON article_section_embedding USING hnsw (embedding vector_ ...
[OK] executed: DROP INDEX IF EXISTS idx_protocol_embedding_embedding; ...
[OK] executed: CREATE INDEX IF NOT EXISTS idx_protocol_embedding_hnsw ON protocol_embedding USING hnsw (embedding vector_cosine_ops) WI ...

✅ Done: HNSW index migration completed.


In [4]:
# analyze_and_explain.py
import os
import psycopg2

def get_dsn():
    dsn = os.getenv("POSTGRES_DSN")
    if dsn:
        return dsn

    host = os.getenv("POSTGRES_HOST", "localhost")
    port = os.getenv("POSTGRES_PORT", "5432")
    db   = os.getenv("POSTGRES_DB", "postgres")
    user = os.getenv("POSTGRES_USER", "postgres")
    pw   = os.getenv("POSTGRES_PASSWORD", "postgres")
    return f"host={host} port={port} dbname={db} user={user} password={pw}"


def run_analyze_and_explain():
    dsn = get_dsn()
    conn = psycopg2.connect(dsn)
    conn.autocommit = False

    try:
        with conn.cursor() as cur:
            print("▶ ANALYZE article_section_embedding / protocol_embedding ...")
            cur.execute("ANALYZE article_section_embedding;")
            cur.execute("ANALYZE protocol_embedding;")
            print("✅ ANALYZE done.\n")

            # ------------------------------
            # EXPLAIN for article_section_embedding (HNSW)
            # ------------------------------
            print("▶ EXPLAIN article_section_embedding (HNSW index 사용 여부 확인)")
            cur.execute("""
                SET LOCAL hnsw.ef_search = 100;

                EXPLAIN (ANALYZE, BUFFERS)
                SELECT chunk_id
                FROM article_section_embedding
                ORDER BY embedding <=> (
                    SELECT embedding FROM article_section_embedding LIMIT 1
                )
                LIMIT 10;
            """)
            rows = cur.fetchall()
            print("---- article_section_embedding ----")
            for (line,) in rows:
                print(line)
            print()

            # ------------------------------
            # EXPLAIN for protocol_embedding (HNSW)
            # ------------------------------
            print("▶ EXPLAIN protocol_embedding (HNSW index 사용 여부 확인)")
            cur.execute("""
                SET LOCAL hnsw.ef_search = 100;

                EXPLAIN (ANALYZE, BUFFERS)
                SELECT chunking_id
                FROM protocol_embedding
                ORDER BY embedding <=> (
                    SELECT embedding FROM protocol_embedding LIMIT 1
                )
                LIMIT 10;
            """)
            rows = cur.fetchall()
            print("---- protocol_embedding ----")
            for (line,) in rows:
                print(line)

        conn.commit()
        print("\n✅ Done: ANALYZE + EXPLAIN finished.")

    except Exception as e:
        conn.rollback()
        print("\n❌ Failed, rolled back.")
        raise
    finally:
        conn.close()


if __name__ == "__main__":
    run_analyze_and_explain()


▶ ANALYZE article_section_embedding / protocol_embedding ...
✅ ANALYZE done.

▶ EXPLAIN article_section_embedding (HNSW index 사용 여부 확인)
---- article_section_embedding ----
Limit  (cost=0.01..0.02 rows=1 width=40) (actual time=0.845..0.846 rows=0.00 loops=1)
  Buffers: shared hit=3
  InitPlan 1
    ->  Limit  (cost=0.00..0.00 rows=1 width=32) (never executed)
          ->  Seq Scan on article_section_embedding article_section_embedding_1  (cost=0.00..0.00 rows=1 width=32) (never executed)
  ->  Sort  (cost=0.01..0.02 rows=1 width=40) (actual time=0.727..0.728 rows=0.00 loops=1)
        Sort Key: ((article_section_embedding.embedding <=> (InitPlan 1).col1))
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=3
        ->  Seq Scan on article_section_embedding  (cost=0.00..0.00 rows=1 width=40) (actual time=0.009..0.009 rows=0.00 loops=1)
Planning:
  Buffers: shared hit=32 read=12 dirtied=1
Planning Time: 2.666 ms
Execution Time: 0.910 ms

▶ EXPLAIN protocol_embedding

In [5]:
import pandas as pd

path = r"C:\dev\study\skn18_fianl-2team\SKN18-FINAL-2TEAM\neo4j\import\ts_embedding_v2.csv"

df = pd.read_csv(path)

# text_chunk 길이 계산
lengths = df["text_chunk"].astype(str).str.len()

# 1000자 이상인 케이스 개수
over_1000 = (lengths >= 1500).sum()

print(f"전체 row 수: {len(df)}")
print(f"텍스트 길이 1000자 이상 row 수: {over_1000}")


전체 row 수: 120277
텍스트 길이 1000자 이상 row 수: 97


In [6]:
import pandas as pd

path = r"C:\dev\study\skn18_fianl-2team\SKN18-FINAL-2TEAM\neo4j\import\ts_embedding_v2.csv"
df = pd.read_csv(path)

# 길이 계산
lengths = df["text_chunk"].astype(str).str.len()

print("=== 기본 통계 ===")
print(lengths.describe())   # 평균, 표준편차, 사분위수 등

# 구간별 분포 (예: 0~200, 200~400, ..., 1800 이상)
bins = [0, 200, 400, 600, 800, 1000, 1200, 1500, 1800, 100000]
labels = ["0-200","200-400","400-600","600-800","800-1000",
        "1000-1200","1200-1500","1500-1800","1800+"]

bucket = pd.cut(lengths, bins=bins, labels=labels, right=False)
dist = bucket.value_counts().sort_index()

print("\n=== 길이 구간별 개수 ===")
print(dist)

# 필요하면 비율도
print("\n=== 길이 구간별 비율 ===")
print((dist / len(df) * 100).round(2).astype(str) + "%")


=== 기본 통계 ===
count    120277.000000
mean        502.828296
std         127.452447
min          49.000000
25%         453.000000
50%         514.000000
75%         562.000000
max        9838.000000
Name: text_chunk, dtype: float64

=== 길이 구간별 개수 ===
text_chunk
0-200         1366
200-400      14096
400-600      99135
600-800       4491
800-1000       773
1000-1200      218
1200-1500      101
1500-1800       36
1800+           61
Name: count, dtype: int64

=== 길이 구간별 비율 ===
text_chunk
0-200         1.14%
200-400      11.72%
400-600      82.42%
600-800       3.73%
800-1000      0.64%
1000-1200     0.18%
1200-1500     0.08%
1500-1800     0.03%
1800+         0.05%
Name: count, dtype: object
